In [1]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
import re
path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 120
dim = 4
gamma = 1e-07
relax_steps = 200


# Pattern to match all relevant CSV files
file_pattern = path + f"Susceptibility_v3_multi_gammas_{dim}_*_per_gamma_20_anisotropy_1.5_gamma_{gamma}_ref*.csv"



# List all matching files
csv_files = glob.glob(file_pattern)

# Print how many files were found
print(f"Found {len(csv_files)} files.")
repeats = len(csv_files)

for f in csv_files:
    print(f)

# Read and concatenate all files
df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df_avg = (
        df_all.groupby(["gamma", "Ht", "direction"], as_index=False)
        .agg(
            chi_mean=("chi", "mean"),
            chi_std=("chi", lambda x: x.std(ddof=1) / np.sqrt(len(x)))
        )
    )

fig_ref = px.line(
    df_avg,
    x="Ht",
    y="chi_mean",
    error_y="chi_std",
    color="direction",  # Different color for up/down
    line_dash="gamma",  # Optional: if you vary gamma too
    markers=True,
    labels={
        "Ht": "Ht (T)",
        "chi_mean": "Susceptibility χ",
        "direction": "Sweep Direction",
    },
    title="Susceptibility χ vs Ht (sweep up vs down)",
    template="plotly_white"
)

fig_ref.show()

Found 2 files.
/Users/jiakai/Desktop/SURF/code/_spirit/Susceptibility_v3_multi_gammas_4_2400_per_gamma_20_anisotropy_1.5_gamma_1e-07_ref_2.csv
/Users/jiakai/Desktop/SURF/code/_spirit/Susceptibility_v3_multi_gammas_4_2400_per_gamma_20_anisotropy_1.5_gamma_1e-07_ref_1.csv


In [2]:
#Relaxed Ht = 4T
import pandas as pd
import glob
import numpy as np
import plotly.express as px
path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
# n_cycles = 240


# Pattern to match all relevant CSV files
file_pattern = path + f"Susceptibility_v3_multi_gammas_{dim}_*_per_gamma_20_anisotropy_1.5_relax_step_200_gammas_-5_relax_4.0_gamma_{gamma}_relaxed_*.csv"

# List all matching files
csv_files = glob.glob(file_pattern)

# Print how many files were found
print(f"Found {len(csv_files)} files.")
repeats = len(csv_files)
for f in csv_files:
    print(f)

# Read and concatenate all files
df_all_relaxed = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

# Average over cycles, std divided by sqrt(n_cycles)
df_avg_relaxed = (
    df_all_relaxed.groupby(["gamma", "Ht"], as_index=False)
    .agg(
        chi_mean=("chi", "mean"),
        chi_std=("chi", lambda x: x.std(ddof=1) / np.sqrt(len(x)))
    )
)

# Plot
fig_relaxed = px.line(
    df_avg_relaxed,
    x="Ht",
    y="chi_mean",
    error_y="chi_std",
    color="gamma",  # Different color per gamma
    markers=True,
    labels={
        "Ht": "Ht (T)",
        "chi_mean": "Susceptibility χ",
        "gamma": "Gamma",
    },
    title="Susceptibility χ vs Ht",
    template="plotly_white"
)


fig_relaxed.show()

Found 1 files.
/Users/jiakai/Desktop/SURF/code/_spirit/Susceptibility_v3_multi_gammas_10_120_per_gamma_20_anisotropy_1.5_relax_step_200_gammas_-5_relax_4.0_gamma_1e-07_relaxed_1.csv


In [3]:
import plotly.graph_objects as go

# Create a new figure
fig = go.Figure()

# --- Plot relaxed data ---
for gamma, df_group in df_avg_relaxed.groupby("gamma"):
    fig.add_trace(
        go.Scatter(
            x=df_group["Ht"],
            y=df_group["chi_mean"],
            error_y=dict(type="data", array=df_group["chi_std"]),
            mode="lines+markers",
            name=f"H_relax = 4.0T",
            line=dict(dash="solid"),
            marker=dict(symbol="circle"),
        )
    )


# # --- Plot relaxed data ---
# for gamma, df_group in df_avg_relaxed_small.groupby("gamma"):
#     fig.add_trace(
#         go.Scatter(
#             x=df_group["Ht"],
#             y=df_group["chi_mean"],
#             error_y=dict(type="data", array=df_group["chi_std"]),
#             mode="lines+markers",
#             name=f"H_relax = 3.2T",
#             line=dict(dash="solid"),
#             marker=dict(symbol="circle"),
#         )
#     )



# # --- Plot relaxed data ---
# for gamma, df_group in df_avg_relaxed_small_1.groupby("gamma"):
#     fig.add_trace(
#         go.Scatter(
#             x=df_group["Ht"],
#             y=df_group["chi_mean"],
#             error_y=dict(type="data", array=df_group["chi_std"]),
#             mode="lines+markers",
#             name=f"H_relax = 2.0T",
#             line=dict(dash="solid"),
#             marker=dict(symbol="circle"),
#         )
#     )


# # --- Plot relaxed data ---
# for gamma, df_group in df_avg_relaxed_tgt.groupby("gamma"):
#     fig.add_trace(
#         go.Scatter(
#             x=df_group["Ht"],
#             y=df_group["chi_mean"],
#             error_y=dict(type="data", array=df_group["chi_std"]),
#             mode="lines+markers",
#             name=f"Multiple",
#             line=dict(dash="solid"),
#             marker=dict(symbol="circle"),
#         )
#     )



# --- Plot sweep data (with direction) ---
for (direction, gamma), df_group in df_avg.groupby(["direction", "gamma"]):
    fig.add_trace(
        go.Scatter(
            x=df_group["Ht"],
            y=df_group["chi_mean"],
            error_y=dict(type="data", array=df_group["chi_std"]),
            mode="lines+markers",
            name=f"{direction}",
            line=dict(dash="solid"),
            marker=dict(symbol="square"),
        )
    )

# Final layout
fig.update_layout(
    title=f"Susceptibility χ vs Ht, γ={gamma}, dim = {dim}",
    xaxis_title="Ht (T)",
    yaxis_title="Susceptibility χ",
    template="plotly_white",
    legend_title="Dataset",
)

fig.write_html(path+ f'susceptibility_v3_{dim}_{gamma}.html')


In [4]:
fig.show()